In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Load Bitcoin data
df = pd.read_csv('../raw_data/Bitcoin_history_data.csv')

# Explore
print(df.shape)
print(df.head())
print(df.info())
print(df.describe())

(4175, 6)
         Date       Close        High         Low        Open    Volume
0  2014-09-17  457.334015  468.174011  452.421997  465.864014  21056800
1  2014-09-18  424.440002  456.859985  413.104004  456.859985  34483200
2  2014-09-19  394.795990  427.834991  384.532013  424.102997  37919700
3  2014-09-20  408.903992  423.295990  389.882996  394.673004  36863600
4  2014-09-21  398.821014  412.425995  393.181000  408.084991  26580100
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4175 entries, 0 to 4174
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    4175 non-null   object 
 1   Close   4175 non-null   float64
 2   High    4175 non-null   float64
 3   Low     4175 non-null   float64
 4   Open    4175 non-null   float64
 5   Volume  4175 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 195.8+ KB
None
               Close           High            Low           Open  \
count    4175.0000

In [2]:
# calculated
df['MA_7'] = df['Close'].rolling(window=7).mean()  # average 7days
df['MA_21'] = df['Close'].rolling(window=21).mean()  # average 21 days
df['Price_Change'] = df['Close'].pct_change()  # average dayly
df['Volatility'] = df['Close'].rolling(window=7).std()  # Volatility 7 days
df['Volume_Change'] = df['Volume'].pct_change()  # Variation volum

# Delete NaN
df = df.dropna()

print(df.head(25))

          Date       Close        High         Low        Open    Volume  \
20  2014-10-07  336.187012  339.247009  320.481995  330.584015  49199900   
21  2014-10-08  352.940002  354.364014  327.187988  336.115997  54736300   
22  2014-10-09  365.026001  382.726013  347.687012  352.747986  83641104   
23  2014-10-10  361.562012  375.066986  352.963013  364.687012  43665700   
24  2014-10-11  362.299011  367.191010  355.950989  361.362000  13345200   
25  2014-10-12  378.549011  379.433014  356.144012  362.605988  17552800   
26  2014-10-13  390.414001  397.226013  368.897003  377.920990  35221400   
27  2014-10-14  400.869995  411.697998  391.324005  391.691986  38491500   
28  2014-10-15  394.773010  402.226990  388.765991  400.954987  25267100   
29  2014-10-16  382.556000  398.807007  373.070007  394.518005  26990000   
30  2014-10-17  383.757996  385.477997  375.389008  382.756012  13600700   
31  2014-10-18  391.441986  395.157990  378.971008  383.976013  11416800   
32  2014-10-

In [3]:
# Target: Pricec in 5 Days
df["future_close"] = df["Close"].shift(-5)
df["target"] = (df["future_close"] > df["Close"]).astype(int)
df = df.dropna()

# Control the distribution of the target
print("Distribution target:")
print(df["target"].value_counts())
print(f"% classe 1: {df['target'].mean():.2%}")

Distribution target:
target
1    2266
0    1884
Name: count, dtype: int64
% classe 1: 54.60%


In [4]:
# Features
features = ["Open", "High", "Low", "Close", "Volume",
            "MA_7", "MA_21", "Price_Change", "Volatility", "Volume_Change"]

X = df[features]
y = df["target"]

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

Shape X: (4150, 10)
Shape y: (4150,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # shuffle=False
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

Train size: (3320, 10)
Test size: (830, 10)


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

print("✅ Model trained successfully!")

✅ Model trained successfully!


In [7]:
from sklearn.metrics import accuracy_score, classification_report

# Predictions
predictions = model.predict(X_test_scaled)

# Accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print(f"John's accuracy: 0.47")
print(f"Amélioration: {accuracy - 0.47:.4f}")

# Report
print("\n" + classification_report(y_test, predictions))

Accuracy: 0.4807
John's accuracy: 0.47
Amélioration: 0.0107

              precision    recall  f1-score   support

           0       0.48      0.99      0.64       392
           1       0.77      0.02      0.04       438

    accuracy                           0.48       830
   macro avg       0.62      0.51      0.34       830
weighted avg       0.63      0.48      0.33       830



In [8]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Predictions
rf_predictions = rf_model.predict(X_test_scaled)

# Accuracy
rf_accuracy = accuracy_score(y_test, rf_predictions)
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Improvement vs John: {rf_accuracy - 0.47:.4f}")

Random Forest Accuracy: 0.5145
Improvement vs John: 0.0445


In [ ]:

from xgboost import XGBClassifier

# XGBoost model
xgb_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_scaled, y_train)

# Predictions
xgb_predictions = xgb_model.predict(X_test_scaled)

# Accuracy
xgb_accuracy = accuracy_score(y_test, xgb_predictions)
print(f"XGBoost Accuracy: {xgb_accuracy:.4f}")
print(f"Improvement vs John: {xgb_accuracy - 0.47:.4f}")

XGBoost Accuracy: 0.5181
Improvement vs John: 0.0481


In [ ]:
# Summary of results
results = {
    "John (Logistic Regression)": 0.47,
    "Nicolas (Logistic + Features)": accuracy,
    "Random Forest": rf_accuracy,
    "XGBoost": xgb_accuracy
}

print("\n=== RESULTS ===")
for model, acc in results.items():
    print(f"{model:30s}: {acc:.4f}")

print(f"\nBest model: {max(results, key=results.get)}")
print(f"Best accuracy: {max(results.values()):.4f}")


=== RESULTS ===
John (Logistic Regression)    : 0.4700
Marcel (Logistic + Features)  : 0.4807
Random Forest                 : 0.5145
XGBoost                       : 0.5181

Best model: XGBoost
Best accuracy: 0.5181
